# OMIn GraphRAG : 1-Setup, 2-Data Inspection & 3-records construction
## OMIn dataset : 
OMIn (Operations & Maintenance Intelligence) is a dataset of real aviation maintenance and incident records from the FAA (Federal Aviation Administration). These are reports filed when something goes wrong with an aircraft: a brake fails, an engine quits, a component is found defective during inspection. Each record describes what happened, what the problem was, and sometimes what action was taken (basically this particular prob/action is present in mainnet data).
## Summary of this notebook : 
*We built records.json, the single source of truth for our entire system.==> This file contains 8917 records from both faa and "clean" mainnet*

### Source 1 : FAA narratives (2748 records, Maintenance_Text_data_nona.csv)
This is the main FAA incident database. It has 181 columns, most of them coded (c5, c119, etc.) because it's an old FAA database format where column names are internal codes. After inspecting the data, we identified that c119 is the narrative text column, the free-text description a pilot or mechanic wrote about the incident. Examples: "TAILWHEEL COCKED RIGHT PRIOR TO TKOF" or "ENGINE QUIT AFTER TAKEOFF". The other columns (c1–c10 etc.) are structured metadata: date, aircraft type, severity code, etc. We kept those as metadata because they'll enrich the graph nodes later (e.g. knowing an incident happened in 1975 vs 1999, or involved a rotary wing vs fixed wing aircraft).
### Source 2 : Aircraft Annotation records (6169 records, Aircraft_Annotation_DataFile.csv)
Someone already structured these records into PROBLEM and ACTION pairs — "ENGINE IDLE OVERRIDE KILLED ENGINE" → "REMOVED & REPLACED FUEL SERVO". This is almost a pre-made triplet: (engine idle override) --[causes]--> (engine failure) and (technician) --[replaced]--> (fuel servo). These will produce much higher quality triplets than the raw FAA narratives because the structure is already there.
### Why combine both?
More records = richer graph = better coverage of the aviation maintenance domain = better answers to queries.
*This file is the backbone of the GraphRAG pointer system. Every graph node and edge will carry record_id references back to this file. When the graph retrieval finds a relevant subgraph, it looks up those record IDs here and fetches the full original text to feed to the LLM. The graph navigates, the records provide the actual knowledge.*


## Installations

In [ ]:
!pip install google-generativeai networkx sentence-transformers pandas tqdm matplotlib pyvis

In [ ]:
!pip install -q google-genai

## Setup

In [ ]:
import os
from google import genai
GEMINI_API_KEY = ""
client = genai.Client(api_key=GEMINI_API_KEY)

## OMIn data inspection

In [ ]:
import pandas as pd
DATA_DIR = "./data"
faa_df = pd.read_csv(f"{DATA_DIR}/Maintenance_Text_data_nona.csv")
print("ispecting FAA file : Maintenance_Text_data_nona")
print(f"Shape: {faa_df.shape}")
print(f"Columns: {list(faa_df.columns)}")
faa_df.head(3) # ==> Shape: (2748, 181) , nb of present records : each has 181 feature 

In [ ]:
#inspecting FAA 100-record sample
faa_100 = pd.read_csv(f"{DATA_DIR}/FAA_sample_100.csv")
print("FAA_sample_100")
print(f"Shape: {faa_100.shape}")
print(f"Columns: {list(faa_100.columns)}")
faa_100.head(3) #==> Shape: (100, 34) : less features 

In [ ]:
#Inspecting mainnet data
import os
maintnet_files = [f for f in os.listdir(DATA_DIR) if 'aint' in f and f.endswith('.csv')]
print(f"MaintNet files : {maintnet_files}")
maintnet_dfs = {}
for f in maintnet_files:
    df = pd.read_csv(f"{DATA_DIR}/{f}")
    maintnet_dfs[f] = df
    print(f"\n{f}: shape={df.shape}, columns={list(df.columns)}")


## Identifying text columns 
From these records, the triplets could be extracted

In [ ]:
# for col in faa_df.columns:
#     if faa_df[col].dtype == object:
#         sample = faa_df[col].dropna().iloc[:2].tolist()
#         avg_len = faa_df[col].dropna().apply(len).mean()
#         print(f"\nColumn '{col}' (avg length: {avg_len:.0f} chars):")
#         for s in sample:
#             print(f"  > {s[:300]}")

In [ ]:
# Check c119 in the full dataset
print("Column c119 sample") #from previious , it contains the main text, making sure 
print(faa_df['c119'].dropna().head(5).tolist())
print(f"\nNon-null count in c119: {faa_df['c119'].notna().sum()}") #==> Non-null count in c119: 2748 ==> "clean"
print(f"Data type: {faa_df['c119'].dtype}")

# #checking columns that have content
# print("\n\nColumns with most non-null values (top 20):")
# non_null = faa_df.notna().sum().sort_values(ascending=False)
# print(non_null.head(20))

In [ ]:
import json, os

#inspecting MaintNet reference files
for fname in ['Aircraft_Annotation_DataFile.csv', 'Aviation_Abbreviation_Dataset.csv', 
              'Aviation_TermBanks_Dataset.csv', 'Aviation_grammar_Dataset.csv']:
    df = pd.read_csv(f"./data/{fname}")
    print(f"\n******{fname}********")
    print(f"Shape: {df.shape}")
    print(f"Columns: {list(df.columns)}")
    print(df.head(3).to_string())

# ==> ******Aircraft_Annotation_DataFile.csv******** Shape: (6169, 3) that has id + problem + action ==> additional info ; clean/annotated data for triplet extraction

In [ ]:
import json, os
os.makedirs("./output", exist_ok=True)

# check abbreviations
abbrev_df = pd.read_csv("./data/Aviation_Abbreviation_Dataset.csv")
abbrev_map = {}
for _, row in abbrev_df.iterrows():
    abbr = str(row['Abbreviated']).strip().upper()
    full = str(row['Standard_Description']).strip()
    abbrev_map[abbr] = full

print(f"Loaded {len(abbrev_map)} abbreviations")

#debugging : expanding abbreviations
import re
def expand_abbreviations(text, abbrev_map):# this function is needed to expand abbreviation befor triplets extraction for better graph quality 
    if not isinstance(text, str):
        return ""
    tokens = text.upper().split()
    expanded = [abbrev_map.get(t, t) for t in tokens]
    return " ".join(expanded) 
#test
sample = "ACFT WAS TAXIING FOR TKOF WHEN PLT NOTED SOFT BRAKE"
print(f"\nOriginal:  {sample}")
print(f"Expanded:  {expand_abbreviations(sample, abbrev_map)}")
#load domain terms for prompt injection
terms_df = pd.read_csv("./data/Aviation_TermBanks_Dataset.csv")
domain_terms = terms_df['Word'].tolist()
print(f"\nLoaded {len(domain_terms)} domain terms: {domain_terms[:10]}")

# Debug: what abbreviations are actually in the map?
print("Sample abbreviation entries:")
for k, v in list(abbrev_map.items())[:20]:
    print(f"  '{k}' → '{v}'")

#check specifically for our test words
for word in ['ACFT', 'PLT', 'TKOF', 'APCH', 'GR', 'FLT']:
    print(f"  {word} in map: {abbrev_map.get(word, 'NOT FOUND')}") #some of these words where not found ==> we benefit from what we have while letting LLM handles the missing ones during extraction

## building unified records 
we have two main text sources which are : 
* FAA c119 narratives : 2748 records (abbreviated, free text) ==> the free-text description a pilot or mechanic wrote about the incident
* Aircraft_Annotation_DataFile problem + action formatted 

In [ ]:
import json, os
os.makedirs("./output", exist_ok=True)

records = []
#FAA narrative
#keep useful metadata columns we know exist ==> structured metadata: date, aircraft type, severity code, etc. keeping them because they'll enrich the graph nodes later
meta_cols = ['c5', 'c1', 'c2', 'c3', 'c4', 'c6', 'c7', 'c8', 'c9', 'c10']

for idx, row in faa_df.iterrows():
    text = str(row['c119']).strip()
    if len(text) < 10:
        continue
    #apply what abbreviations we have, l
    expanded = expand_abbreviations(text, abbrev_map)
    records.append({
        "record_id": f"FAA_{idx}",
        "text_original": text,
        "text": expanded,
        "source": "FAA_narratives",
        "source_file": "Maintenance_Text_data_nona.csv",
        "metadata": {col: str(row[col]) for col in meta_cols if col in faa_df.columns}
    })

print(f"FAA narrative records: {len(records)}")

# Aircraft Annotation records (PROBLEM + ACTION)
ann_df = pd.read_csv("./data/Aircraft_Annotation_DataFile.csv")
start = len(records)

for idx, row in ann_df.iterrows():
    problem = str(row['PROBLEM']).strip() if pd.notna(row['PROBLEM']) else ""
    action = str(row['ACTION']).strip() if pd.notna(row['ACTION']) else ""
    if len(problem) < 5:
        continue
    combined = f"PROBLEM: {problem} ACTION: {action}" if action else problem
    records.append({
        "record_id": f"ANN_{row['IDENT']}",
        "text_original": combined,
        "text": expand_abbreviations(combined, abbrev_map),
        "source": "Aircraft_Annotation",
        "source_file": "Aircraft_Annotation_DataFile.csv",
        "metadata": {
            "ident": str(row['IDENT']),
            "problem": problem,
            "action": action
        }
    })

print(f"Annotation records added: {len(records) - start}")
print(f"Total records: {len(records)}")

#save
with open("./output/records.json", "w") as f:
    json.dump(records, f, indent=2)

print("\nSample FAA record:")
print(json.dumps(records[0], indent=2))
print("\nSample Annotation record:")
print(json.dumps(records[2750], indent=2))